# 05 — Аудиопрофили пользователей

`a_bar_u` — среднее аудиоэмбеддингов треков из train-истории пользователя,
только по items с непустым эмбеддингом. Логика — в
`research/scripts/build_audio.py`; `--profiles-only` не трогает 13.8 ГБ parquet.

In [ ]:
# Colab: раскомментировать. Локально ячейка не нужна.
# from google.colab import userdata
# token = userdata.get('git')
# !git clone -q https://$token@github.com/Vladislavbro/music-recommendations.git
# %cd music-recommendations
# !pip install -q uv && uv pip install --system -e ".[research]"


In [ ]:
CONFIG = 'research/configs/aggregators_50m.yaml'

!uv run python research/scripts/build_audio.py --config {CONFIG} --profiles-only


## Sanity: пересчитать профиль одного юзера вручную

In [ ]:
import pickle
import random
from pathlib import Path

import numpy as np

from grouprec.config import build, load_config
from grouprec.data.splits import SplitConfig, global_temporal_split
from grouprec.data.yambda_loader import DataConfig, apply_item_remap, prepare_interactions

ARTIFACTS = Path.cwd() / 'artifacts'
cfg = load_config(CONFIG)

embeds = np.load(ARTIFACTS / 'audio' / 'embeddings.npy')
profiles = np.load(ARTIFACTS / 'audio' / 'user_profiles.npy')
user_audio_valid = np.load(ARTIFACTS / 'audio' / 'user_audio_valid.npy')
with open(ARTIFACTS / 'audio' / 'uid_to_row.pkl', 'rb') as f:
    uid_to_row = pickle.load(f)
with open(ARTIFACTS / 'gsasrec' / 'item_id_to_idx.pkl', 'rb') as f:
    item_id_to_idx = pickle.load(f)

df = apply_item_remap(prepare_interactions(build(DataConfig, cfg['data'])), item_id_to_idx)
train_df, _, _ = global_temporal_split(df, build(SplitConfig, cfg['split']))

audio_valid = np.linalg.norm(embeds, axis=1) > 0
uid = random.Random(0).choice([u for u, ok in zip(uid_to_row, user_audio_valid) if ok])
items = train_df.loc[train_df['uid'] == uid, 'item_idx'].to_numpy()
expected = embeds[items[audio_valid[items]]].mean(axis=0)
diff = float(np.abs(expected - profiles[uid_to_row[uid]]).max())
print(f'uid={uid}, история={len(items)}, max|diff|={diff:.2e}')
assert diff < 1e-5
